In [ ]:
# config bootstrap (auto-added): resolve repo paths from config.py
import os as _os, sys as _sys
_h = _os.path.abspath(_os.getcwd())
while not _os.path.exists(_os.path.join(_h, 'config.py')) and _os.path.dirname(_h) != _h:
    _h = _os.path.dirname(_h)
_sys.path.insert(0, _h)
import config as _cfg

# Day 1 — Evidence Feature Engineering

Goal: push from 87.4% toward 89-90% by extracting more from existing data.

**Two improvements based on RED-DOT findings:**
1. **Aggregation features** — instead of just mean(s2_list), add max/min/std for each evidence type. Catches the "single most relevant evidence" signal that RED-DOT identified.
2. **Element-wise interactions** — products and differences between signals, exploiting cross-feature relationships XGBoost can leverage.

**Important:** This requires going back to Cell 5 of the evidence pipeline and saving per-evidence-image scores (lists), not just means. We'll need to recompute, but it's fast — Wikipedia and downloads are cached, only CLIP scoring runs.

**Two-step plan:**
- Step A — Re-extract evidence scores keeping per-image granularity (saves to `evidence_clip_scores_v2.csv`)
- Step B — Build expanded feature matrix and retrain XGBoost

In [3]:
# ── Step A — Re-score evidence with per-image granularity ──
# Modified from evidence_pipeline_4060_v2.ipynb Cell 5
# Difference: stores list of per-image scores instead of mean only
# Uses same downloader (parallel, timeout=3) and same CLIP

import os
import json
import numpy as np
import pandas as pd
import torch
import requests
import clip
from PIL import Image
from io import BytesIO
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT   = str(_cfg.ROOT)
DATASET_ROOT   = os.path.join(PROJECT_ROOT, 'kaggle_dataset_full')
CLIP_FEATURES  = os.path.join(PROJECT_ROOT, 'clip_finetuned_v2', 'val_features')
CLIP_MODEL_DIR = os.path.join(PROJECT_ROOT, 'models', 'clip')
LINKS_FILE     = os.path.join(PROJECT_ROOT, 'links_val.json')
VAL_ANN_PATH   = os.path.join(DATASET_ROOT, 'merged_balanced', 'val.json')
VAL_META_PATH  = os.path.join(DATASET_ROOT, 'metadata', 'val.json')
SCORES_V2_CACHE = os.path.join(PROJECT_ROOT, 'evidence_clip_scores_v2.csv')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# ── Load IDs and metadata ──
id_df = pd.read_csv(os.path.join(CLIP_FEATURES, 'val_sample_ids.csv'))
id_df['id'] = id_df['id'].astype(str)
labels = id_df['label'].values

with open(LINKS_FILE, 'r') as f:
    links_data = json.load(f)
with open(VAL_ANN_PATH, 'r', encoding='utf-8') as f:
    ann_data = json.load(f)
annotations = ann_data['annotations']
idx_to_id = {str(i): str(a['id']) for i, a in enumerate(annotations)}
id_to_links = {idx_to_id[idx]: links_data[idx] for idx in links_data if idx in idx_to_id}

with open(VAL_META_PATH, 'r', encoding='utf-8') as f:
    metadata = json.load(f)

print(f'Val samples: {len(id_df)}')
print(f'With links : {sum(1 for sid in id_df["id"] if sid in id_to_links)}/{len(id_df)}')

Device: cuda
Val samples: 5000
With links : 4967/5000


In [4]:
# ── Load CLIP ──
print('Loading CLIP ViT-L/14...')
os.environ['CLIP_DOWNLOAD_ROOT'] = CLIP_MODEL_DIR
clip_model, clip_preprocess = clip.load('ViT-L/14', device=device, jit=False)
clip_model.eval()
clip_model = clip_model.float()
for p in clip_model.parameters():
    p.requires_grad = False

def get_image_features(image_pil):
    with torch.no_grad():
        img_tensor = clip_preprocess(image_pil).unsqueeze(0).to(device)
        features   = clip_model.encode_image(img_tensor)
        features   = features / features.norm(dim=-1, keepdim=True).clamp(min=1e-8)
    return features.cpu().numpy()[0]

def get_text_features(text):
    with torch.no_grad():
        tokens   = clip.tokenize([text], truncate=True).to(device)
        features = clip_model.encode_text(tokens)
        features = features / features.norm(dim=-1, keepdim=True).clamp(min=1e-8)
    return features.cpu().numpy()[0]

def cosine_sim(a, b):
    return float(np.dot(a, b))

def resolve_image_path(rel_path):
    return os.path.join(DATASET_ROOT, str(rel_path).replace('visual_news/', 'images/'))

# ── Parallel downloader ──
HEADERS       = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
MAX_EV_IMAGES = 3
TIMEOUT       = 3
MAX_WORKERS   = 6

def download_image(url):
    try:
        resp = requests.get(url, timeout=TIMEOUT, headers=HEADERS)
        if resp.status_code == 200 and len(resp.content) > 1000:
            img = Image.open(BytesIO(resp.content)).convert('RGB')
            if img.size[0] > 50 and img.size[1] > 50:
                return img
    except:
        pass
    return None

def download_urls_parallel(url_list, max_images):
    results = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(download_image, url): url for url in url_list[:max_images*3]}
        for future in as_completed(futures):
            img = future.result()
            if img:
                results.append(img)
            if len(results) >= max_images:
                break
    return results

def get_evidence_images(links):
    direct_urls = [(lp[0] if isinstance(lp, list) else lp) 
                    for lp in links.get('links_direct_search', [])]
    inv_urls    = [(lp[0] if isinstance(lp, list) else lp) 
                    for lp in links.get('links_inv_search', [])]
    return download_urls_parallel(direct_urls, MAX_EV_IMAGES), \
           download_urls_parallel(inv_urls,    MAX_EV_IMAGES)

print('Setup complete.')

Loading CLIP ViT-L/14...
Setup complete.


In [5]:
# ── Re-score loop with per-image storage ──
# Stores RAW per-image scores as JSON strings, then we aggregate later
# Same parallel/cache logic as before

SAVE_EVERY = 50

if os.path.exists(SCORES_V2_CACHE):
    scores_df = pd.read_csv(SCORES_V2_CACHE)
    scores_df['id'] = scores_df['id'].astype(str)
    already_scored = set(scores_df['id'].values)
    print(f'Resuming — {len(already_scored)} already scored')
else:
    scores_df = pd.DataFrame()
    already_scored = set()
    print('Starting fresh')

remaining = [sid for sid in id_df['id'].values if sid not in already_scored]
new_results = []

print(f'Samples to score: {len(remaining)}')

for i, sample_id in enumerate(tqdm(remaining, desc='Re-scoring')):
    try:
        if sample_id not in metadata:
            new_results.append({'id': sample_id, 's2_list': '[]', 's3_list': '[]', 
                                's4_list': '[]', 's5_list': '[]', 's6_list': '[]',
                                'n_direct': 0, 'n_inverse': 0})
            continue

        meta     = metadata[sample_id]
        caption  = meta['caption']
        img_path = resolve_image_path(meta['image_path'])
        orig_img = Image.open(img_path).convert('RGB')

        orig_feat    = get_image_features(orig_img)
        caption_feat = get_text_features(caption)

        links = id_to_links.get(sample_id, {})
        direct_imgs, inv_imgs = get_evidence_images(links)

        # Direct evidence
        s2_list, s3_list, direct_feats = [], [], []
        for ev_img in direct_imgs:
            ev_feat = get_image_features(ev_img)
            direct_feats.append(ev_feat)
            s2_list.append(cosine_sim(orig_feat, ev_feat))
            s3_list.append(cosine_sim(caption_feat, ev_feat))

        # Inverse evidence
        s4_list, s5_list, inv_feats = [], [], []
        for ev_img in inv_imgs:
            ev_feat = get_image_features(ev_img)
            inv_feats.append(ev_feat)
            s4_list.append(cosine_sim(orig_feat, ev_feat))
            s5_list.append(cosine_sim(caption_feat, ev_feat))

        # Cross-evidence
        s6_list = []
        for df in direct_feats:
            for ivf in inv_feats:
                s6_list.append(cosine_sim(df, ivf))

        new_results.append({
            'id'       : sample_id,
            's2_list'  : json.dumps(s2_list),
            's3_list'  : json.dumps(s3_list),
            's4_list'  : json.dumps(s4_list),
            's5_list'  : json.dumps(s5_list),
            's6_list'  : json.dumps(s6_list),
            'n_direct' : len(direct_imgs),
            'n_inverse': len(inv_imgs),
        })

    except Exception as e:
        new_results.append({'id': sample_id, 's2_list': '[]', 's3_list': '[]',
                            's4_list': '[]', 's5_list': '[]', 's6_list': '[]',
                            'n_direct': 0, 'n_inverse': 0})

    if len(new_results) % SAVE_EVERY == 0:
        partial = pd.DataFrame(new_results)
        combined = pd.concat([scores_df, partial], ignore_index=True) if not scores_df.empty else partial
        combined.to_csv(SCORES_V2_CACHE, index=False)

# Final save
final = pd.DataFrame(new_results)
if not scores_df.empty:
    final = pd.concat([scores_df, final], ignore_index=True)
final.to_csv(SCORES_V2_CACHE, index=False)
print(f'Done. {len(final)} samples scored.')

Starting fresh
Samples to score: 5000


Re-scoring:   1%|          | 32/5000 [04:06<10:37:43,  7.70s/it]


KeyboardInterrupt: 

In [ ]:
# ── Step B — Build expanded feature matrix from per-image scores ──
# Loads everything fresh, builds engineered features, retrains XGBoost

import os, json
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, roc_auc_score

PROJECT_ROOT  = str(_cfg.ROOT)
CLIP_FEATURES = os.path.join(PROJECT_ROOT, 'clip_finetuned_v2', 'val_features')

# ── Base signals ──
clip_probs = np.load(os.path.join(CLIP_FEATURES, 'clip_finetuned_probs.npy'))
clip_sims  = np.load(os.path.join(CLIP_FEATURES, 'clip_finetuned_sims.npy'))
id_df      = pd.read_csv(os.path.join(CLIP_FEATURES, 'val_sample_ids.csv'))
id_df['id'] = id_df['id'].astype(str)
labels     = id_df['label'].values

deb_df       = pd.read_csv(os.path.join(PROJECT_ROOT, 'deberta_val_scores_v2.csv'))
deb_df['id'] = deb_df['id'].astype(str)
deb_lookup   = dict(zip(deb_df['id'], deb_df['entailment_score']))
deb_scores   = np.array([deb_lookup.get(i, 0.33) for i in id_df['id']])

wiki_df       = pd.read_csv(os.path.join(PROJECT_ROOT, 'wiki_nli_scores.csv'))
wiki_df['id'] = wiki_df['id'].astype(str)
wiki_lookup   = {row['id']: row for _, row in wiki_df.iterrows()}

# ── Per-image evidence scores ──
ev_df = pd.read_csv(os.path.join(PROJECT_ROOT, 'evidence_clip_scores_v2.csv'))
ev_df['id'] = ev_df['id'].astype(str)
ev_lookup = {row['id']: row for _, row in ev_df.iterrows()}

def parse_list(s):
    try:    return json.loads(s)
    except: return []

def aggregate_signal(score_list, default=0.0):
    """Build mean/max/min/std/count from a list of scores."""
    if not score_list:
        return [default, default, default, 0.0, 0]
    arr = np.array(score_list)
    return [arr.mean(), arr.max(), arr.min(), arr.std() if len(arr)>1 else 0.0, len(arr)]

# ── Build per-sample aggregations ──
rows = []
for sid in id_df['id'].values:
    ev = ev_lookup.get(sid, {})
    s2 = parse_list(ev.get('s2_list', '[]'))
    s3 = parse_list(ev.get('s3_list', '[]'))
    s4 = parse_list(ev.get('s4_list', '[]'))
    s5 = parse_list(ev.get('s5_list', '[]'))
    s6 = parse_list(ev.get('s6_list', '[]'))
    
    row = (
        aggregate_signal(s2) + aggregate_signal(s3) +
        aggregate_signal(s4) + aggregate_signal(s5) +
        aggregate_signal(s6)
    )
    rows.append(row)

ev_features = np.array(rows)  # shape (5000, 25) — 5 stats × 5 signals
print(f'Evidence aggregation matrix: {ev_features.shape}')

# Column names for inspection
stat_names = ['mean', 'max', 'min', 'std', 'count']
sig_names = ['s2', 's3', 's4', 's5', 's6']
ev_cols = [f'{s}_{stat}' for s in sig_names for stat in stat_names]
print(f'Evidence features: {ev_cols}')

In [ ]:
# ── Add element-wise interactions ──
# RED-DOT finding: element-wise products + differences boost accuracy 8.9% over plain concat

# Pull max-aggregations for each evidence signal (single most-relevant-evidence theory)
s2_max = ev_features[:, 1]   # s2_max
s3_max = ev_features[:, 6]   # s3_max
s4_max = ev_features[:, 11]  # s4_max
s5_max = ev_features[:, 16]  # s5_max
s6_max = ev_features[:, 21]  # s6_max

s2_mean = ev_features[:, 0]
s3_mean = ev_features[:, 5]
s4_mean = ev_features[:, 10]
s5_mean = ev_features[:, 15]

# Wiki signals
w1 = np.array([wiki_lookup.get(i, {}).get('wiki_score',     0.33) for i in id_df['id']])
w2 = np.array([wiki_lookup.get(i, {}).get('wiki_score_min', 0.33) for i in id_df['id']])
w3 = np.array([wiki_lookup.get(i, {}).get('wiki_score_max', 0.33) for i in id_df['id']])
w4 = np.array([wiki_lookup.get(i, {}).get('entity_count',   0)    for i in id_df['id']])
w5 = np.array([wiki_lookup.get(i, {}).get('wiki_coverage',  0)    for i in id_df['id']])

# Element-wise interactions inspired by RED-DOT
# These capture the ACTUAL out-of-context signal:
# - When fake: orig image matches evidence (s2 high) but caption doesn't (s3 low)
diff_image_text_direct = s2_max - s3_max     # high = visual match without text match
diff_image_text_inv    = s4_max - s5_max
ratio_text_image_direct = s3_max / (s2_max + 1e-6)
ratio_text_image_inv    = s5_max / (s4_max + 1e-6)

# CLIP-evidence agreement
clip_s2_product  = clip_probs * s2_max     # interaction: fine-tuned CLIP × max evidence
clip_s3_product  = clip_probs * s3_max
clip_diff        = clip_probs - clip_sims  # disagreement between two CLIP heads

# Cross-modal evidence consistency
s6_above_avg     = s6_max - s2_mean        # cross-evidence stronger than direct?

# ── Final feature matrix ──
X_eng = np.column_stack([
    # Base signals
    clip_probs, clip_sims, deb_scores,
    # Wiki (5)
    w1, w2, w3, w4, w5,
    # Evidence aggregations (25)
    ev_features,
    # Element-wise interactions (8)
    diff_image_text_direct, diff_image_text_inv,
    ratio_text_image_direct, ratio_text_image_inv,
    clip_s2_product, clip_s3_product, clip_diff,
    s6_above_avg,
])

feature_names = (
    ['clip_prob', 'clip_sim', 'deberta'] +
    ['wiki_mean', 'wiki_min', 'wiki_max', 'entity_cnt', 'wiki_cov'] +
    ev_cols +
    ['s2_minus_s3', 's4_minus_s5', 's3_div_s2', 's5_div_s4',
     'clip_x_s2', 'clip_x_s3', 'clip_diff', 's6_above_s2_mean']
)

print(f'Final feature matrix: {X_eng.shape}')
print(f'Feature count: {len(feature_names)}')

In [ ]:
# ── Train XGBoost with engineered features ──
X_train, X_val, y_train, y_val = train_test_split(
    X_eng, labels, test_size=0.2, random_state=42, stratify=labels)

xgb = XGBClassifier(
    n_estimators=1000,    # higher cap with early stopping
    max_depth=5,           # one deeper than before — more features = more depth
    learning_rate=0.03,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=2,
    reg_alpha=0.1,
    reg_lambda=1.0,
    eval_metric='logloss',
    early_stopping_rounds=50,
    random_state=42,
)

xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=100)

preds = xgb.predict(X_val)
probs = xgb.predict_proba(X_val)[:, 1]

acc = accuracy_score(y_val, preds)
f1  = f1_score(y_val, preds)
auc = roc_auc_score(y_val, probs)

print()
print('=' * 65)
print('XGBoost with engineered features')
print('=' * 65)
print(f'Accuracy : {acc*100:.2f}%')
print(f'F1       : {f1:.4f}')
print(f'AUC      : {auc:.4f}')
print()
print(classification_report(y_val, preds, target_names=['REAL', 'FAKE']))

In [ ]:
# ── Feature importance — see which engineered features actually help ──
import pandas as pd

imp_df = pd.DataFrame({
    'feature'   : feature_names,
    'importance': xgb.feature_importances_
}).sort_values('importance', ascending=False)

print('Top 20 features:')
print(imp_df.head(20).to_string(index=False))
print()
print('Bottom 10 features (potentially droppable):')
print(imp_df.tail(10).to_string(index=False))

# Group importance by category
print()
print('Importance by category:')
categories = {
    'CLIP'         : ['clip_prob', 'clip_sim'],
    'Text (deb/wiki)': ['deberta', 'wiki_mean', 'wiki_min', 'wiki_max', 'entity_cnt', 'wiki_cov'],
    'Evidence agg' : ev_cols,
    'Interactions' : ['s2_minus_s3', 's4_minus_s5', 's3_div_s2', 's5_div_s4',
                       'clip_x_s2', 'clip_x_s3', 'clip_diff', 's6_above_s2_mean'],
}
for cat, feats in categories.items():
    total = imp_df[imp_df['feature'].isin(feats)]['importance'].sum()
    print(f'  {cat:<20}: {total:.4f}')

In [ ]:
# ── Comparison to baseline ──
print('=' * 65)
print('PROGRESSION')
print('=' * 65)
results = [
    ('CLIP alone (v2)',                 85.6),
    ('+ DeBERTa + Evidence (mean only)',87.4),
    ('+ Wiki NLI',                      87.2),
    ('+ Engineered features (Day 1)',  acc * 100),
    ('--- Targets ---',                 0),
    ('SNIFFER',                         88.4),
    ('MUSE-MLP',                        90.0),
    ('RED-DOT',                         90.3),
    ('MUSE-AITR',                       93.3),
]
for name, val in results:
    if val == 0:
        print(f'  {name}')
    else:
        marker = ' <-- YOU' if 'Day 1' in name else ''
        print(f'  {name:<40} {val:.2f}%{marker}')